# Surrogate Model Retraining

Use this notebook to retrain the surrogate contact-pressure model whenever new simulation results are available.

## When to retrain
- After running a new batch of FEBio simulations that cover new catheter designs, speeds, or tissue properties
- When prediction accuracy on new cases degrades noticeably
- Periodically (e.g., monthly) as the simulation database grows

## Workflow overview
1. **Discover** new `.xplt` results not yet in the training CSV
2. **Extract** surrogate data from new cases and append to `combined.csv`
3. **Retrain** on the full combined dataset (or fine-tune on new data only)
4. **Evaluate** the new model vs. the previous best
5. **Deploy** if the new model is better (copy artifacts to `models/latest/`)

> **Tip:** Run `full_pipeline.ipynb` instead if you are starting from scratch with no trained model.

## 0. Environment Check

In [ ]:
import sys, os
import importlib

# Verify required packages
missing = []
for pkg in ["xplt_core", "surrogate_lab", "torch", "mlflow", "pandas", "numpy", "sklearn", "yaml"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"WARNING: Missing packages: {missing}")
    print("Run: bash scripts/setup-common-env.sh && source .venv/bin/activate")
else:
    import torch, mlflow
    print(f"Python: {sys.version.split()[0]}")
    print(f"PyTorch: {torch.__version__}")
    print(f"MLflow: {mlflow.__version__}")
    print("All packages OK")

## 1. Configuration

Adjust paths below to match your environment.

In [ ]:
from pathlib import Path
import os

# ── Paths ──────────────────────────────────────────────────────────────────
# Root of the simulation-api-tool repo (auto-detected)
REPO_ROOT = Path(os.environ.get("REPO_ROOT", Path.cwd().parent))

# Directory containing FEBio simulation outputs (.xplt files)
RUNS_DIR = Path(os.environ.get("RUNS_DIR", REPO_ROOT / "runs"))

# Surrogate data directory (bind-mounted to /app/surrogate_data in Docker)
SURROGATE_DATA_DIR = Path(os.environ.get("SURROGATE_DATA_PATH", REPO_ROOT / "data" / "surrogate"))
TRAINING_DIR   = SURROGATE_DATA_DIR / "training"
MODELS_DIR     = SURROGATE_DATA_DIR / "models" / "latest"
RESULTS_DIR    = SURROGATE_DATA_DIR / "results"

# MLflow tracking server
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")

# Combined training CSV (all cases merged)
COMBINED_CSV      = TRAINING_DIR / "combined.csv"
REFERENCE_FACETS  = TRAINING_DIR / "reference_facets.csv"

# Previous model directory (for comparison)
PREV_MODELS_DIR   = SURROGATE_DATA_DIR / "models" / "previous"

# Ensure directories exist
for d in [TRAINING_DIR, MODELS_DIR, RESULTS_DIR, PREV_MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"RUNS_DIR          : {RUNS_DIR}  (exists: {RUNS_DIR.exists()})")
print(f"SURROGATE_DATA_DIR: {SURROGATE_DATA_DIR}")
print(f"COMBINED_CSV      : {COMBINED_CSV}  (exists: {COMBINED_CSV.exists()})")
print(f"MODELS_DIR        : {MODELS_DIR}")
print(f"MLFLOW_URI        : {MLFLOW_URI}")

## 2. Discover New Simulation Cases

Find `.xplt` files that have not yet been extracted into the training CSV.

In [ ]:
import pandas as pd
import json

def find_xplt_files(runs_dir: Path) -> list[dict]:
    """Walk runs/ and return metadata for all .xplt files found."""
    cases = []
    for xplt_path in sorted(runs_dir.rglob("*.xplt")):
        case_dir = xplt_path.parent
        run_id = case_dir.name

        # Try to load simulation parameters from params.json if it exists
        params = {}
        params_file = case_dir / "params.json"
        if params_file.exists():
            with open(params_file) as f:
                params = json.load(f)

        cases.append({
            "run_id": run_id,
            "xplt_path": xplt_path,
            "case_dir": case_dir,
            "params": params,
        })
    return cases


all_cases = find_xplt_files(RUNS_DIR)
print(f"Found {len(all_cases)} .xplt files total")

# Determine which run_ids have already been extracted
already_extracted = set()
if COMBINED_CSV.exists():
    existing_df = pd.read_csv(COMBINED_CSV, usecols=["run_id"])
    already_extracted = set(existing_df["run_id"].unique())
    print(f"Already extracted: {len(already_extracted)} run_ids in combined.csv")

new_cases = [c for c in all_cases if c["run_id"] not in already_extracted]
print(f"New cases to extract: {len(new_cases)}")

for c in new_cases[:10]:
    print(f"  {c['run_id']}  →  {c['xplt_path'].name}")

## 3. Extract Surrogate Data from New Cases

Uses `xplt_core.SimulationCase` to extract per-facet contact pressure across all timesteps.

In [ ]:
import xplt_core
from tqdm.auto import tqdm

new_dfs = []
failed_cases = []

for case_info in tqdm(new_cases, desc="Extracting"):
    try:
        case = xplt_core.SimulationCase(case_info["xplt_path"])
        df = case.df_surrogate()
        if df is None or len(df) == 0:
            print(f"  SKIP {case_info['run_id']}: df_surrogate() returned empty")
            continue

        # Add run_id for traceability
        df["run_id"] = case_info["run_id"]

        # Save per-case CSV for inspection
        case_csv = TRAINING_DIR / f"{case_info['run_id']}.csv"
        df.to_csv(case_csv, index=False)

        new_dfs.append(df)
        print(f"  OK {case_info['run_id']}: {len(df):,} rows")

    except Exception as exc:
        print(f"  FAIL {case_info['run_id']}: {exc}")
        failed_cases.append((case_info["run_id"], str(exc)))

print(f"\nExtracted {len(new_dfs)} new cases, {len(failed_cases)} failed")

## 4. Append New Data to combined.csv

In [ ]:
if not new_dfs:
    print("No new data to append.")
    if COMBINED_CSV.exists():
        combined_df = pd.read_csv(COMBINED_CSV)
        print(f"Using existing combined.csv: {len(combined_df):,} rows")
    else:
        raise RuntimeError(
            "No new data AND no existing combined.csv. "
            "Run full_pipeline.ipynb first, or add new simulation results."
        )
else:
    new_combined_df = pd.concat(new_dfs, ignore_index=True)

    if COMBINED_CSV.exists():
        # Load existing and append
        old_df = pd.read_csv(COMBINED_CSV)
        combined_df = pd.concat([old_df, new_combined_df], ignore_index=True)
        print(f"Appended {len(new_combined_df):,} rows to existing {len(old_df):,} rows")
    else:
        combined_df = new_combined_df
        print(f"Created new combined.csv with {len(combined_df):,} rows")

    combined_df.to_csv(COMBINED_CSV, index=False)
    print(f"Saved combined.csv → {COMBINED_CSV}")
    print(f"Total rows: {len(combined_df):,}")

print(f"\nColumns: {list(combined_df.columns)}")
print(combined_df.describe())

## 5. Update reference_facets.csv (Optional)

Only needed if the catheter geometry changed (new .feb design). Skip this cell if retraining on same geometry.

In [ ]:
UPDATE_REFERENCE_FACETS = False  # Set to True if catheter geometry changed

if UPDATE_REFERENCE_FACETS:
    # Use the first (or latest) successfully extracted case for reference geometry
    if not new_dfs:
        print("No new data — skipping reference_facets update.")
    else:
        ref_df = new_dfs[0][["centroid_x", "centroid_y", "centroid_z", "facet_area"]].drop_duplicates()
        ref_df.to_csv(REFERENCE_FACETS, index=False)
        print(f"Updated reference_facets.csv: {len(ref_df):,} facets → {REFERENCE_FACETS}")
else:
    if REFERENCE_FACETS.exists():
        print(f"Keeping existing reference_facets.csv ({pd.read_csv(REFERENCE_FACETS).shape[0]:,} facets)")
    else:
        print("WARNING: reference_facets.csv does not exist — run full_pipeline.ipynb first.")

## 6. Configure Training

Load the surrogate-lab default config and override paths for retraining.

In [ ]:
import yaml
import copy

# ── Find default config from surrogate-lab ────────────────────────────────
SURROGATE_LAB_DIR = REPO_ROOT / "surrogate-lab"
default_cfg_path = SURROGATE_LAB_DIR / "config.yaml"

if not default_cfg_path.exists():
    raise FileNotFoundError(
        f"surrogate-lab config.yaml not found at {default_cfg_path}. "
        "Ensure surrogate-lab is present in the repo root."
    )

with open(default_cfg_path) as f:
    cfg = yaml.safe_load(f)

# Deep copy so we don't mutate the surrogate-lab original
cfg = copy.deepcopy(cfg)

# ── Override paths and URIs ───────────────────────────────────────────────
cfg["data"]             = cfg.get("data", {})
cfg["data"]["source"]   = str(COMBINED_CSV)

cfg["mlflow"]["tracking_uri"]    = MLFLOW_URI
cfg["mlflow"]["experiment_name"] = cfg["mlflow"].get("experiment_name", "contact_pressure_surrogate")
cfg["mlflow"]["log_artifacts"]   = True

CHECKPOINT_DIR = RESULTS_DIR / "retraining_checkpoint"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
cfg["training"]["checkpoint"] = {
    "dir"     : str(CHECKPOINT_DIR),
    "enabled" : True,
    "save_best": True,
}

# ── Auto-tune architecture ─────────────────────────────────────────────────
n_rows = len(combined_df)
if n_rows < 5_000:
    cfg["model"]["layers"] = [32, 16]
elif n_rows < 50_000:
    cfg["model"]["layers"] = [64, 32, 16]
else:
    cfg["model"]["layers"] = [128, 64, 32]

print("Training configuration:")
print(f"  data source  : {cfg['data']['source']}")
print(f"  MLflow URI   : {cfg['mlflow']['tracking_uri']}")
print(f"  experiment   : {cfg['mlflow']['experiment_name']}")
print(f"  architecture : {cfg['model']['layers']}")
print(f"  checkpoint   : {CHECKPOINT_DIR}")

# Save the resolved training config for reproducibility
training_cfg_path = RESULTS_DIR / "retraining_config.yaml"
with open(training_cfg_path, "w") as f:
    yaml.dump(cfg, f)
print(f"\nConfig saved → {training_cfg_path}")

## 7. Retrain the Surrogate Model

Runs the full surrogate-lab training pipeline on the updated combined dataset.

In [ ]:
import sys
sys.path.insert(0, str(SURROGATE_LAB_DIR))

from surrogate_lab.data import load_simulation_data, split_data
from surrogate_lab.features import FeaturePipeline
from surrogate_lab.model import build_model
from surrogate_lab.train import train

# Load and split data
print("Loading data ...")
X_raw, y_raw = load_simulation_data(cfg)
print(f"  {len(X_raw):,} samples, {X_raw.shape[1]} features")

pipeline = FeaturePipeline(cfg)
X, y = pipeline.transform(X_raw, y_raw)

X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, cfg)
print(f"  train={len(X_train):,}  val={len(X_val):,}  test={len(X_test):,}")

# Build model
model = build_model(cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"  Model parameters: {n_params:,}")

# Train
print("\nTraining ...")
run_id = train(model, pipeline, X_train, y_train, X_val, y_val, cfg)
print(f"\nTraining complete. MLflow run_id: {run_id}")

## 8. Evaluate New Model vs Previous

Compute test-set metrics and compare against the previously deployed model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch

model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_pred_norm = model(X_test_t).numpy()

y_pred = pipeline.inverse_transform_y(y_pred_norm)
y_true = pipeline.inverse_transform_y(y_test)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
r2   = r2_score(y_true, y_pred)

print("=" * 40)
print("New model — test-set metrics")
print(f"  RMSE : {rmse:.6f} MPa")
print(f"  MAE  : {mae:.6f} MPa")
print(f"  R²   : {r2:.4f}")
print("=" * 40)

# Compare vs previous deployed model (if available)
prev_metrics_file = PREV_MODELS_DIR / "metrics.json"
if prev_metrics_file.exists():
    import json
    with open(prev_metrics_file) as f:
        prev_metrics = json.load(f)
    print(f"\nPrevious model — RMSE: {prev_metrics.get('rmse', 'N/A'):.6f} MPa")
    improvement = prev_metrics.get('rmse', rmse) - rmse
    print(f"Improvement: {improvement:+.6f} MPa RMSE")
else:
    print("\nNo previous model metrics found — this is the first deployment.")

# Scatter plot
fig, ax = plt.subplots(figsize=(5, 5))
lim = max(y_true.max(), y_pred.max()) * 1.05
ax.scatter(y_true, y_pred, alpha=0.3, s=5)
ax.plot([0, lim], [0, lim], 'r--', lw=1)
ax.set_xlabel("True contact pressure (MPa)")
ax.set_ylabel("Predicted (MPa)")
ax.set_title(f"Retrained model  R²={r2:.3f}")
fig.tight_layout()
plot_path = RESULTS_DIR / "retrain_scatter.png"
fig.savefig(plot_path, dpi=120)
plt.show()
print(f"Plot saved → {plot_path}")

## 9. Deploy New Model

If the new model is better (or no previous model exists), copy artifacts to `models/latest/`.

Set `FORCE_DEPLOY = True` to deploy regardless of metric comparison.

In [ ]:
import shutil
import json

FORCE_DEPLOY = False  # Set True to skip metric comparison

# Decide whether to deploy
should_deploy = FORCE_DEPLOY
if not should_deploy:
    if prev_metrics_file.exists():
        with open(prev_metrics_file) as f:
            prev_metrics = json.load(f)
        should_deploy = rmse < prev_metrics.get("rmse", float("inf"))
        if should_deploy:
            print(f"New RMSE ({rmse:.6f}) < previous ({prev_metrics['rmse']:.6f}) — deploying.")
        else:
            print(f"New RMSE ({rmse:.6f}) >= previous ({prev_metrics['rmse']:.6f}) — NOT deploying.")
            print("Set FORCE_DEPLOY = True to override.")
    else:
        should_deploy = True
        print("No previous model — deploying.")

if should_deploy:
    # Archive the current model before overwriting
    if MODELS_DIR.exists() and any(MODELS_DIR.iterdir()):
        shutil.copytree(MODELS_DIR, PREV_MODELS_DIR, dirs_exist_ok=True)
        print(f"Archived previous model → {PREV_MODELS_DIR}")

    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    # Copy artifacts from checkpoint dir to models/latest/
    for fname in ["best_model.pt", "x_scaler.pkl", "y_scaler.pkl", "config.yaml"]:
        src = CHECKPOINT_DIR / fname
        if src.exists():
            shutil.copy2(src, MODELS_DIR / fname)
            print(f"  Copied {fname}")
        else:
            print(f"  WARNING: {fname} not found in checkpoint dir")

    # Save metrics for future comparison
    metrics_out = {"rmse": float(rmse), "mae": float(mae), "r2": float(r2), "run_id": run_id}
    with open(MODELS_DIR / "metrics.json", "w") as f:
        json.dump(metrics_out, f, indent=2)

    print(f"\nDeployment complete: {MODELS_DIR}")
    print("The API will load the new model on the next prediction request.")
else:
    print("Deployment skipped.")

## 10. Quick Sanity Check

Verify the deployed model loads correctly via `SurrogatePredictor`.

In [ ]:
from digital_twin_ui.surrogate.predictor import SurrogatePredictor, is_model_available
from digital_twin_ui.surrogate.csar_engine import CSAREngine

if is_model_available(MODELS_DIR):
    predictor = SurrogatePredictor(MODELS_DIR)
    print(f"Predictor loaded. Feature names: {predictor.feature_names}")

    # Run a quick CSAR check with reference facets
    if REFERENCE_FACETS.exists():
        engine = CSAREngine(predictor)
        depths = [50.0, 100.0, 150.0]
        bands  = [{"zmin": 0, "zmax": 50, "label": "tip"}]
        df_csar = engine.compute_from_csv(REFERENCE_FACETS, depths, bands)
        print("\nCSAR sanity check:")
        print(df_csar[["insertion_depth_mm", "band_label", "csar", "n_contact_facets"]].to_string(index=False))
    else:
        print("reference_facets.csv not found — skipping CSAR check.")
else:
    print("WARNING: Model artifacts not found at", MODELS_DIR)

## 11. Summary

Print a deployment summary for the record.

In [ ]:
from datetime import datetime

print("=" * 60)
print("RETRAINING SUMMARY")
print(f"  Date            : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  New cases added : {len(new_dfs)}")
print(f"  Failed cases    : {len(failed_cases)}")
print(f"  Total rows      : {len(combined_df):,}")
print(f"  MLflow run_id   : {run_id}")
print(f"  RMSE            : {rmse:.6f} MPa")
print(f"  R²              : {r2:.4f}")
print(f"  Deployed        : {should_deploy}")
if should_deploy:
    print(f"  Model path      : {MODELS_DIR}")
print("=" * 60)

if failed_cases:
    print("\nFailed cases:")
    for run_id_fail, err in failed_cases:
        print(f"  {run_id_fail}: {err}")